# Text Mining for AI — Final Assignment

group 59 

Three tasks: NERC (comparing spaCy vs BERT), sentiment (VADER), and topic (zero-shot)

In [1]:
# !pip install spacy transformers[torch] vaderSentiment pandas scikit-learn
# !python -m spacy download en_core_web_sm

In [2]:
import pandas as pd
import spacy
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline as hf_pipeline
from sklearn.metrics import classification_report
nltk.download('vader_lexicon', quiet=True)

True

## Loading the data

In [3]:
sent_df = pd.read_csv('Sentiment-topic-test.tsv', sep='\t', header=0)
sent_df.columns = ['id', 'text', 'sentiment', 'topic']

ner_df = pd.read_csv('NER-test.tsv', sep='\t', header=0)
ner_df.columns = ['sent_id', 'word_idx', 'token', 'tag']

print(sent_df[['text','sentiment','topic']].to_string())

                                                                                                                                                    text sentiment       topic
0                                                              It took eight years for Warner Brothers to recover from the disaster that was this movie.  negative       movie
1                                                   All the New York University students love this diner in Soho so it makes for a fun young atmosphere.  positive  restaurant
2                                   This Italian place is really trendy but they have forgotten about the most important part of a restaurant, the food.  negative  restaurant
3                                                   In conclusion, my review of this book would be: I like Jane Austen and understand why she is famous.  positive        book
4  The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African American 

In [4]:
# quick look at the NER tag distribution
print(ner_df['tag'].value_counts())

tag
O         183
I-PER       8
B-PER       6
B-ORG       4
B-LOC       4
I-ORG       3
B-MISC      3
I-LOC       2
I-MISC      1
Name: count, dtype: int64


## 1. NERC

We're comparing two systems here. spaCy was the main tool from the labs, and `dslim/bert-base-NER` is a BERT model fine-tuned specifically on CoNLL-2003 so it should be a fair comparison since our test set uses the same annotation scheme.

For evaluation we check whether the predicted entity text and type both match the gold — partial matches don't count.

In [5]:
# get gold entity spans from the BIO tags in the NER file
def bio_to_entities(tokens, tags):
    ents, cur = [], None
    for tok, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if cur: ents.append(cur)
            cur = [tok, tag[2:]]
        elif tag.startswith('I-') and cur:
            cur[0] += ' ' + tok
        else:
            if cur: ents.append(cur)
            cur = None
    if cur: ents.append(cur)
    return [(text, etype) for text, etype in ents]

gold = {}
for sid, grp in ner_df.groupby('sent_id'):
    gold[sid] = bio_to_entities(grp['token'].tolist(), grp['tag'].tolist())

# check
for sid, ents in gold.items():
    if ents:
        print(f"S{sid}: {ents}")

S0: [('Warner Brothers', 'ORG')]
S1: [('New York University', 'ORG'), ('Soho', 'LOC')]
S2: [('Italian', 'MISC')]
S3: [('Jane Austen', 'PER')]
S4: [('Carl Brashear', 'PER'), ('Cuba Gooding Jr.', 'PER'), ('African American', 'MISC'), ('Navy', 'ORG')]
S5: [("Chris O'Donnell", 'PER')]
S6: [('Amsterdam', 'LOC'), ('Blauwbrug', 'ORG')]
S7: [('Dame Maggie Smith', 'PER')]
S8: [('Mr. Kruno', 'PER'), ('New York', 'LOC'), ('Los Angeles', 'LOC')]
S9: [('English', 'MISC')]


### spaCy

spaCy uses OntoNotes labels so we need to remap to the CoNLL types in our test set.

In [6]:
def reconstruct_and_get_offsets(tokens):
    # join tokens with spaces and record each token's char start/end
    offsets = []
    char_pos = 0
    for token in tokens:
        offsets.append((char_pos, char_pos + len(token)))
        char_pos += len(token) + 1  # +1 for the space
    return " ".join(tokens), offsets

nlp = spacy.load('en_core_web_sm')

# spaCy label -> CoNLL label
label_map = {
    'PERSON': 'PER', 'ORG': 'ORG',
    'GPE': 'LOC', 'LOC': 'LOC',
    'NORP': 'MISC', 'WORK_OF_ART': 'MISC',
    'PRODUCT': 'MISC', 'EVENT': 'MISC',
    'FAC': 'MISC', 'LAW': 'MISC', 'LANGUAGE': 'MISC'
}

spacy_preds = {}
for sid, grp in ner_df.groupby('sent_id'):
    tokens = grp['token'].tolist()
    text, offsets = reconstruct_and_get_offsets(tokens)
    doc = nlp(text)

    bio_tags = ['O'] * len(tokens)
    for ent in doc.ents:
        bio_type = label_map.get(ent.label_)
        if bio_type is None:
            continue
        first = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start < ent.end_char and tok_end > ent.start_char:
                bio_tags[i] = f'B-{bio_type}' if first else f'I-{bio_type}'
                first = False

    spacy_preds[sid] = bio_to_entities(tokens, bio_tags)

### BERT (`dslim/bert-base-NER`)

Since this model was fine-tuned on CoNLL-2003 its labels already match our test set, no remapping needed.

In [7]:
bert_ner = hf_pipeline(
    'ner',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple',
    device=-1
)

bert_preds = {}
for sid, grp in ner_df.groupby('sent_id'):
    tokens = grp['token'].tolist()
    text, offsets = reconstruct_and_get_offsets(tokens)
    result = bert_ner(text)

    bio_tags = ['O'] * len(tokens)
    for ent in result:
        first = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start < ent['end'] and tok_end > ent['start']:
                bio_tags[i] = f"B-{ent['entity_group']}" if first else f"I-{ent['entity_group']}"
                first = False

    bert_preds[sid] = bio_to_entities(tokens, bio_tags)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Results

In [8]:
# side by side per sentence
print(f"{'S':<3}  {'gold':<40}  {'spaCy':<40}  {'BERT'}")
print('─' * 120)
for sid in sorted(gold):
    g = ', '.join(f"{t}({e})" for t,e in gold[sid]) or '—'
    s = ', '.join(f"{t}({e})" for t,e in spacy_preds[sid]) or '—'
    b = ', '.join(f"{t}({e})" for t,e in bert_preds[sid]) or '—'
    print(f"S{sid:<2}  {g:<40}  {s:<40}  {b}")

S    gold                                      spaCy                                     BERT
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
S0   Warner Brothers(ORG)                      Warner Brothers(ORG)                      Warner Brothers(ORG)
S1   New York University(ORG), Soho(LOC)       the New York University(ORG), Soho(LOC)   New York University(ORG), Soho(LOC)
S2   Italian(MISC)                             Italian(MISC)                             Italian(MISC)
S3   Jane Austen(PER)                          Jane Austen(PER)                          Jane Austen(PER)
S4   Carl Brashear(PER), Cuba Gooding Jr.(PER), African American(MISC), Navy(ORG)  Carl Brashear(PER), Cuba Gooding Jr.(PER), African American(MISC), Navy(ORG)  Carl Brashear(PER), Cuba Gooding Jr.(PER), African American(MISC), Navy(ORG)
S5   Chris O'Donnell(PER)                      Chris O'Donnell(PER)                      Chris O'Donnel

In [9]:
# P/R/F1 per entity type
def ner_scores(gold_dict, pred_dict):
    types = ['PER', 'ORG', 'LOC', 'MISC']
    tp = {e: 0 for e in types}
    fp = {e: 0 for e in types}
    fn = {e: 0 for e in types}
    for sid in gold_dict:
        g = {(t.lower().strip('.'), et) for t, et in gold_dict[sid]}
        p = {(t.lower().strip('.'), et) for t, et in pred_dict.get(sid, [])}
        for et in types:
            gs = {x for x in g if x[1] == et}
            ps = {x for x in p if x[1] == et}
            tp[et] += len(gs & ps)
            fp[et] += len(ps - gs)
            fn[et] += len(gs - ps)
    print(f"  {'type':<6}  {'TP':>3}  {'FP':>3}  {'FN':>3}  {'P':>6}  {'R':>6}  {'F1':>6}")
    print('  ' + '─'*38)
    total = [0, 0, 0]
    for et in types:
        t, f, n = tp[et], fp[et], fn[et]
        total[0]+=t; total[1]+=f; total[2]+=n
        p_ = t/(t+f) if t+f else 0
        r_ = t/(t+n) if t+n else 0
        f1 = 2*p_*r_/(p_+r_) if p_+r_ else 0
        print(f"  {et:<6}  {t:>3}  {f:>3}  {n:>3}  {p_:>6.3f}  {r_:>6.3f}  {f1:>6.3f}")
    t, f, n = total
    p_ = t/(t+f) if t+f else 0
    r_ = t/(t+n) if t+n else 0
    f1 = 2*p_*r_/(p_+r_) if p_+r_ else 0
    print('  ' + '─'*38)
    print(f"  {'total':<6}  {t:>3}  {f:>3}  {n:>3}  {p_:>6.3f}  {r_:>6.3f}  {f1:>6.3f}")

print('spaCy:')
ner_scores(gold, spacy_preds)
print()
print('BERT:')
ner_scores(gold, bert_preds)

spaCy:
  type     TP   FP   FN       P       R      F1
  ──────────────────────────────────────
  PER       4    2    2   0.667   0.667   0.667
  ORG       2    1    2   0.667   0.500   0.571
  LOC       4    0    0   1.000   1.000   1.000
  MISC      3    1    0   0.750   1.000   0.857
  ──────────────────────────────────────
  total    13    4    4   0.765   0.765   0.765

BERT:
  type     TP   FP   FN       P       R      F1
  ──────────────────────────────────────
  PER       4    2    2   0.667   0.667   0.667
  ORG       3    0    1   1.000   0.750   0.857
  LOC       4    1    0   0.800   1.000   0.889
  MISC      3    0    0   1.000   1.000   1.000
  ──────────────────────────────────────
  total    14    3    3   0.824   0.824   0.824


## 2. Sentiment Analysis

Using VADER here, it's rule-based so no training needed. The compound score goes from -1 to 1; we use the standard thresholds (≥ 0.05 positive, ≤ -0.05 negative, neutral in between).

In [10]:
sia = SentimentIntensityAnalyzer()

def vader_label(text):
    c = sia.polarity_scores(text)['compound']
    return 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')

sent_df['compound'] = sent_df['text'].apply(lambda x: round(sia.polarity_scores(x)['compound'], 3))
sent_df['pred_sentiment'] = sent_df['text'].apply(vader_label)

print(sent_df[['text', 'sentiment', 'pred_sentiment', 'compound']].to_string())

                                                                                                                                                    text sentiment pred_sentiment  compound
0                                                              It took eight years for Warner Brothers to recover from the disaster that was this movie.  negative       negative    -0.625
1                                                   All the New York University students love this diner in Soho so it makes for a fun young atmosphere.  positive       positive     0.818
2                                   This Italian place is really trendy but they have forgotten about the most important part of a restaurant, the food.  negative       positive     0.074
3                                                   In conclusion, my review of this book would be: I like Jane Austen and understand why she is famous.  positive       positive     0.361
4  The story of this movie is focused on Carl Brashear playe

In [11]:
print(classification_report(sent_df['sentiment'], sent_df['pred_sentiment']))

              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.83      0.56      0.56        10
weighted avg       0.80      0.60      0.57        10



## 3. Topic Classification

For topic we use zero-shot classification with facebook/bart-large-mnli. It works by framing each label as a hypothesis and scoring it against the sentence, no training data needed. The candidate labels match the gold labels in the file exactly.

In [12]:
zero_shot = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1
)

LABELS = ['movie', 'restaurant', 'book']

sent_df['pred_topic'] = sent_df['text'].apply(
    lambda x: zero_shot(x, LABELS)['labels'][0]
)

print(sent_df[['text', 'topic', 'pred_topic']].to_string())

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

                                                                                                                                                    text       topic  pred_topic
0                                                              It took eight years for Warner Brothers to recover from the disaster that was this movie.       movie       movie
1                                                   All the New York University students love this diner in Soho so it makes for a fun young atmosphere.  restaurant  restaurant
2                                   This Italian place is really trendy but they have forgotten about the most important part of a restaurant, the food.  restaurant  restaurant
3                                                   In conclusion, my review of this book would be: I like Jane Austen and understand why she is famous.        book        book
4  The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African

In [13]:
print(classification_report(sent_df['topic'], sent_df['pred_topic']))

              precision    recall  f1-score   support

        book       1.00      1.00      1.00         2
       movie       1.00      1.00      1.00         5
  restaurant       1.00      1.00      1.00         3

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



## Visualizations

In [16]:
pip install plotly kaleido

  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached kaleido-1.3.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached narwhals-2.21.2-py3-none-any.whl.metadata (16 kB)
  Using cached choreographer-1.3.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached logistro-2.0.1-py3-none-any.whl.metadata (3.9 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached kaleido-1.3.0-py3-none-any.whl (55 kB)
Using cached choreographer-1.3.0-py3-none-any.whl (52 kB)
Using cached logistro-2.0.1-py3-none-any.whl (8.6 kB)
Using cached narwhals-2.21.2-py3-none-any.whl (451 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [kaleido]m4/7 [plotly]s]
Note: you may need to restart the kernel to use updated packages.


In [17]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from sklearn.metrics import confusion_matrix

# ── helper: recompute F1 per entity type ──────────────────────
def get_f1s(gold_dict, pred_dict):
    types = ['PER', 'ORG', 'LOC', 'MISC']
    out = {}
    tot = [0, 0, 0]
    for et in types:
        tp = fp = fn = 0
        for sid in gold_dict:
            g = {(t.lower().strip('.'), e) for t, e in gold_dict[sid] if e == et}
            p = {(t.lower().strip('.'), e) for t, e in pred_dict.get(sid, []) if e == et}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        tot[0] += tp; tot[1] += fp; tot[2] += fn
        pr = tp/(tp+fp) if tp+fp else 0
        re = tp/(tp+fn) if tp+fn else 0
        out[et] = round(2*pr*re/(pr+re) if pr+re else 0, 3)
    tp, fp, fn = tot
    pr = tp/(tp+fp) if tp+fp else 0
    re = tp/(tp+fn) if tp+fn else 0
    out['total'] = round(2*pr*re/(pr+re) if pr+re else 0, 3)
    return out

# ── 1. NER F1 grouped bar ─────────────────────────────────────
spacy_f1 = get_f1s(gold, spacy_preds)
bert_f1  = get_f1s(gold, bert_preds)
type_labels = ['PER', 'ORG', 'LOC', 'MISC', 'total']

ner_plot_df = pd.DataFrame({
    'type':   type_labels * 2,
    'F1':     [spacy_f1[l] for l in type_labels] + [bert_f1[l] for l in type_labels],
    'system': ['spaCy'] * 5 + ['BERT'] * 5
})

fig1 = px.bar(
    ner_plot_df, x='type', y='F1', color='system', barmode='group',
    title='NER — F1 per entity type (spaCy vs BERT)',
    range_y=[0, 1], text='F1',
    color_discrete_map={'spaCy': '#4c72b0', 'BERT': '#ee8434'}
)
fig1.update_traces(textposition='outside', texttemplate='%{text:.2f}')
fig1.update_layout(yaxis_title='F1', xaxis_title='', legend_title='')
fig1.show()
fig1.write_image('ner_f1.png')

# ── 2. Sentiment confusion matrix heatmap ─────────────────────
sent_labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(sent_df['sentiment'], sent_df['pred_sentiment'], labels=sent_labels)

fig2 = px.imshow(
    cm,
    x=sent_labels, y=sent_labels,
    labels=dict(x='predicted', y='gold', color='count'),
    title='VADER sentiment — confusion matrix',
    text_auto=True, color_continuous_scale='Blues'
)
fig2.show()
fig2.write_image('sentiment_cm.png')

# ── 3. Zero-shot confidence per sentence ──────────────────────
zs_results  = [zero_shot(t, LABELS) for t in sent_df['text']]
top_scores  = [r['scores'][0] for r in zs_results]
top_preds   = [r['labels'][0] for r in zs_results]
short_texts = [t[:50] + '…' if len(t) > 50 else t for t in sent_df['text']]

zs_df = pd.DataFrame({
    'text':       short_texts,
    'confidence': top_scores,
    'topic':      top_preds
})

fig3 = px.bar(
    zs_df, x='confidence', y='text', orientation='h',
    color='topic', title='Zero-shot topic — confidence per sentence',
    range_x=[0, 1], text='confidence',
    color_discrete_map={'movie': '#4c72b0', 'restaurant': '#ee8434', 'book': '#2ca02c'}
)
fig3.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig3.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title='confidence', yaxis_title='')
fig3.show()
fig3.write_image('topic_confidence.png')